# Z-FMN full study on RAGTruth (v3)

All 1,350 RAGTruth test answers from Mistral-7B, Llama-2-7B and Llama-2-13B, with and without hallucinations.
Each answer gets: token log-probs, 10 resamples, the Z-FMN network, an entailment check against the source,
and the SelfCheckGPT baseline.

**This takes several Kaggle sessions.** Every session does about 8.5 hours of new work, saves it,
and the next session carries on from there. You do not edit anything between sessions.

**Settings (once):** Accelerator → GPU T4 x2, Internet on, Add-ons → Secrets → `HF_TOKEN` attached.
Also click "Request access" on the meta-llama/Llama-2-13b-chat-hf page with the same Hugging Face account.

**Inputs (the right-hand panel → Add Input → Your Work):**
1. Session 1: add the old notebook that ran Run 2 and Run 3, so its answers are reused instead of regenerated.
2. Every later session: also add *this* notebook (its latest version), so the previous session's work is reused.

Then press **Save Version → Save & Run All (Commit)** and close the tab.


In [ ]:
!nvidia-smi -L
import os
WORK = '/kaggle/working/zfmn' if os.path.exists('/kaggle') else '/content/zfmn'
os.makedirs(WORK + '/zfmn', exist_ok=True)
os.makedirs(WORK + '/tests', exist_ok=True)
%cd $WORK

In [ ]:
!pip install -q accelerate bitsandbytes spacy networkx transformers statsmodels wordfreq
!pip install -q --no-deps emoatlas
!python -m spacy download en_core_web_sm -q

## Code
These cells write the Z-FMN package to disk. Just run them.

In [ ]:
%%writefile zfmn/__init__.py
"""Z-FMN: fuzzy-augmented forma mentis networks with node- and edge-level epistemic confidence."""

__version__ = "0.1.0"


In [ ]:
%%writefile zfmn/data.py
"""RAGTruth loading. RAGTruth (Niu et al. 2024) has word-level human hallucination spans."""

import json
import os
import urllib.request

RAGTRUTH_URL = "https://raw.githubusercontent.com/ParticleMedia/RAGTruth/main/dataset/{name}"
FILES = ("response.jsonl", "source_info.jsonl")

# RAGTruth model label -> Hugging Face model id. Only open-weight generators are usable,
# because we score each response with the model that wrote it.
OPEN_MODELS = {
    "mistral-7B-instruct": "mistralai/Mistral-7B-Instruct-v0.1",
    "llama-2-7b-chat": "meta-llama/Llama-2-7b-chat-hf",
    "llama-2-13b-chat": "meta-llama/Llama-2-13b-chat-hf",
}


def download_ragtruth(data_dir):
    os.makedirs(data_dir, exist_ok=True)
    for name in FILES:
        path = os.path.join(data_dir, name)
        if not os.path.exists(path):
            print(f"downloading {name}")
            urllib.request.urlretrieve(RAGTRUTH_URL.format(name=name), path)


def source_text(src):
    """The grounding material the model was given, as plain text."""
    info = src["source_info"]
    if isinstance(info, str):
        return info
    if src["task_type"] == "QA":
        return info.get("passages", "")
    return json.dumps(info, ensure_ascii=False)


def load_ragtruth(data_dir, model_label, split="test", tasks=None, limit=None, labelled_only=False):
    """Return a list of dicts: id, prompt, response, spans [(start, end)], task_type."""
    download_ragtruth(data_dir)
    sources = {}
    with open(os.path.join(data_dir, "source_info.jsonl")) as f:
        for line in f:
            d = json.loads(line)
            sources[d["source_id"]] = d
    rows = []
    with open(os.path.join(data_dir, "response.jsonl")) as f:
        for line in f:
            r = json.loads(line)
            if r["model"] != model_label or r["split"] != split:
                continue
            src = sources[r["source_id"]]
            if tasks and src["task_type"] not in tasks:
                continue
            if labelled_only and not r["labels"]:
                continue
            rows.append({
                "id": r["id"],
                "task_type": src["task_type"],
                "prompt": src["prompt"],
                "response": r["response"],
                "spans": [(lab["start"], lab["end"]) for lab in r["labels"]],
                "span_types": [lab["label_type"] for lab in r["labels"]],
                "source_text": source_text(src),
            })
    rows.sort(key=lambda x: int(x["id"]))
    return rows[:limit] if limit else rows


In [ ]:
%%writefile zfmn/lm.py
"""Token-level uncertainty from a causal LM: teacher-forced scoring and sampling."""

import torch


def load_model(name, load_in_4bit=False, device=None):
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(name, use_fast=True)
    if not tok.is_fast:
        raise ValueError("A fast tokenizer is required for character offsets.")
    kwargs = {}
    if load_in_4bit:
        from transformers import BitsAndBytesConfig

        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16
        )
        kwargs["device_map"] = "auto"
    elif torch.cuda.is_available():
        kwargs["torch_dtype"] = torch.float16
        kwargs["device_map"] = "auto"
    model = AutoModelForCausalLM.from_pretrained(name, **kwargs)
    if device and not load_in_4bit:
        model.to(device)
    model.eval()
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return model, tok


def format_prompt(tok, prompt):
    """Wrap the user prompt in the model's chat template when it has one."""
    if getattr(tok, "chat_template", None):
        return tok.apply_chat_template(
            [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
        )
    return prompt + "\n\n"


@torch.no_grad()
def score_response(model, tok, prompt, response, max_length=4096):
    """Teacher-force `response` after `prompt`.

    Returns one dict per response token: char offsets (start, end) into `response`,
    log-probability of the token, and entropy of the predictive distribution.
    """
    prompt_ids = tok(format_prompt(tok, prompt), add_special_tokens=False)["input_ids"]
    enc = tok(response, add_special_tokens=False, return_offsets_mapping=True)
    resp_ids, offsets = enc["input_ids"], enc["offset_mapping"]
    if not resp_ids:
        return []
    keep = max_length - len(resp_ids)
    if keep < 1:
        raise ValueError("Response longer than max_length.")
    prompt_ids = prompt_ids[-keep:]  # truncate the prompt from the left if needed
    ids = torch.tensor([prompt_ids + resp_ids], device=model.device)
    logits = model(ids).logits[0].float()
    # position i predicts token i+1; response tokens start at len(prompt_ids)
    pred = logits[len(prompt_ids) - 1 : len(prompt_ids) - 1 + len(resp_ids)]
    logp = torch.log_softmax(pred, dim=-1)
    target = torch.tensor(resp_ids, device=logp.device)
    tok_lp = logp.gather(1, target[:, None])[:, 0]
    entropy = -(logp.exp() * logp).sum(-1)
    return [
        {"start": int(s), "end": int(e), "logprob": float(lp), "entropy": float(h)}
        for (s, e), lp, h in zip(offsets, tok_lp.tolist(), entropy.tolist())
    ]


@torch.no_grad()
def sample_responses(model, tok, prompt, n, temperature=0.7, max_new_tokens=300, batch_size=5):
    """Draw n sampled responses to `prompt` (texts only; score them with score_response)."""
    text = format_prompt(tok, prompt)
    ids = tok(text, add_special_tokens=False, return_tensors="pt")["input_ids"].to(model.device)
    out = []
    while len(out) < n:
        k = min(batch_size, n - len(out))
        gen = model.generate(
            ids,
            do_sample=True,
            temperature=temperature,
            top_p=1.0,
            max_new_tokens=max_new_tokens,
            num_return_sequences=k,
            pad_token_id=tok.pad_token_id,
        )
        for g in gen:
            out.append(tok.decode(g[ids.shape[1]:], skip_special_tokens=True).strip())
    return out


In [ ]:
%%writefile zfmn/network.py
"""Textual forma mentis network (TFMN) construction with word-level confidence.

Mirrors the EmoAtlas construction (content words linked when they are within K hops in the
dependency tree) but keeps character offsets and dependency paths, which EmoAtlas does not
return and which the token-to-concept alignment needs.
"""

import math
from collections import defaultdict

import networkx as nx

CONTENT_POS = {"NOUN", "PROPN", "VERB", "ADJ", "ADV", "NUM"}

_NLP = None


def get_nlp(model="en_core_web_sm"):
    global _NLP
    if _NLP is None:
        import spacy

        _NLP = spacy.load(model)
    return _NLP


def word_confidence(doc, token_stats):
    """Aggregate sub-word token stats to spaCy words (Technical Challenge 1, step 1).

    For each spaCy token returns: joint log-prob (sum), mean log-prob, min log-prob,
    mean entropy and conf = exp(mean log-prob), the geometric-mean token probability in [0, 1].
    """
    out = []
    j = 0
    n = len(token_stats)
    for w in doc:
        s, e = w.idx, w.idx + len(w.text)
        # LM offsets are sorted; skip tokens that end before this word
        while j < n and token_stats[j]["end"] <= s:
            j += 1
        k, lps, hs = j, [], []
        while k < n and token_stats[k]["start"] < e:
            if token_stats[k]["end"] > s:
                lps.append(token_stats[k]["logprob"])
                hs.append(token_stats[k]["entropy"])
            k += 1
        if lps:
            mean_lp = sum(lps) / len(lps)
            out.append({
                "joint_lp": sum(lps), "mean_lp": mean_lp, "min_lp": min(lps),
                "entropy": sum(hs) / len(hs), "conf": math.exp(mean_lp),
            })
        else:
            out.append(None)
    return out


def is_content(w, include_numbers=True):
    if w.is_stop or w.is_punct or w.is_space:
        return False
    if w.pos_ == "NUM" or w.like_num:
        return include_numbers
    return w.pos_ in CONTENT_POS and w.is_alpha


def node_key(w):
    return w.text if (w.like_num or w.pos_ == "NUM") else w.lemma_.lower()


def build_tfmn(text, token_stats=None, max_distance=3, include_numbers=True):
    """Build one text's TFMN.

    Returns {"nodes": {key: [occurrence]}, "edges": {(a, b): [occurrence]}} where a node
    occurrence holds its char span and word stats, and an edge occurrence holds the char spans
    of both endpoints and the confidence of the weakest word on its dependency path
    (Technical Challenge 1, step 3).
    """
    doc = get_nlp()(text)
    wstats = word_confidence(doc, token_stats) if token_stats is not None else [None] * len(doc)
    nodes, edges = defaultdict(list), defaultdict(list)
    for sent in doc.sents:
        g = nx.Graph()
        for w in sent:
            g.add_node(w.i)
            if w.head.i != w.i:
                g.add_edge(w.i, w.head.i)
        content = [w for w in sent if is_content(w, include_numbers)]
        for w in content:
            nodes[node_key(w)].append({
                "span": (w.idx, w.idx + len(w.text)), "stats": wstats[w.i],
            })
        idx = {w.i for w in content}
        for w in content:
            paths = nx.single_source_shortest_path(g, w.i, cutoff=max_distance)
            for v_i, path in paths.items():
                if v_i <= w.i or v_i not in idx:
                    continue
                v = doc[v_i]
                a, b = node_key(w), node_key(v)
                if a == b:
                    continue
                confs = [wstats[i]["conf"] for i in path if wstats[i] is not None]
                key = (a, b) if a < b else (b, a)
                edges[key].append({
                    "spans": [(w.idx, w.idx + len(w.text)), (v.idx, v.idx + len(v.text))],
                    "path_conf": min(confs) if confs else None,
                    "distance": len(path) - 1,
                })
    return {"nodes": dict(nodes), "edges": dict(edges)}


In [ ]:
%%writefile zfmn/zops.py
"""Hybrid Z-numbers: fuzzy value A plus a reliability B derived from a subjective-logic opinion.

B is combined along paths with subjective-logic discounting and across sources with cumulative
fusion; A uses triangular fuzzy numbers. Kang et al. (2012) converts Z = (A, B) to a single
fuzzy number for ranking.
"""

import math
from dataclasses import dataclass

import networkx as nx


@dataclass(frozen=True)
class Opinion:
    b: float  # belief
    d: float  # disbelief
    u: float  # uncertainty
    a: float = 0.5  # base rate

    @property
    def projected(self):
        return self.b + self.a * self.u


def from_evidence(r, s, W=2.0, a=0.5):
    """Opinion from positive evidence r and negative evidence s (beta-distribution mapping)."""
    r, s = max(r, 0.0), max(s, 0.0)
    k = r + s + W
    return Opinion(r / k, s / k, W / k, a)


def discount(o1, o2):
    """Trust discounting along i -> j -> k: reliability can only drop along a path."""
    return Opinion(o1.b * o2.b, o1.b * o2.d, o1.d + o1.u + o1.b * o2.u, o2.a)


def fuse(o1, o2):
    """Cumulative fusion of two independent opinions about the same element."""
    den = o1.u + o2.u - o1.u * o2.u
    if den == 0:  # both dogmatic
        return Opinion((o1.b + o2.b) / 2, (o1.d + o2.d) / 2, 0.0, (o1.a + o2.a) / 2)
    b = (o1.b * o2.u + o2.b * o1.u) / den
    u = o1.u * o2.u / den
    den_a = o1.u + o2.u - 2 * o1.u * o2.u
    a = (
        (o1.a * o2.u + o2.a * o1.u - (o1.a + o2.a) * o1.u * o2.u) / den_a
        if den_a != 0 else (o1.a + o2.a) / 2
    )
    return Opinion(b, 1 - b - u, u, a)


@dataclass(frozen=True)
class TFN:
    """Triangular fuzzy number (l, m, r)."""
    l: float
    m: float
    r: float

    @property
    def centroid(self):
        return (self.l + self.m + self.r) / 3

    def scale(self, k):
        return TFN(self.l * k, self.m * k, self.r * k)


def tfn_from_values(values):
    """(min, median, max) of observed values."""
    v = sorted(values)
    n = len(v)
    med = v[n // 2] if n % 2 else (v[n // 2 - 1] + v[n // 2]) / 2
    return TFN(v[0], med, v[-1])


def reliability_tfn(o):
    """B as a fuzzy number: support [belief, plausibility], peak at the projected probability."""
    return TFN(o.b, o.projected, o.b + o.u)


@dataclass(frozen=True)
class ZNumber:
    A: TFN
    B: TFN

    def kang(self):
        """Kang et al. (2012): weight A by sqrt of B's centroid to get one fuzzy number."""
        return self.A.scale(math.sqrt(max(self.B.centroid, 0.0)))


# ---- Z-weighted network measures ---------------------------------------------------------


def to_graph(nodes, edges):
    """nodes: {key: (ZNumber, Opinion)}, edges: {(a, b): (ZNumber, Opinion)} -> nx.Graph."""
    g = nx.Graph()
    for k, (z, o) in nodes.items():
        g.add_node(k, z=z, op=o)
    for (a, b), (z, o) in edges.items():
        g.add_edge(a, b, z=z, op=o,
                   rel_cost=-math.log(max(o.b, 1e-9)),
                   dist=max(1.0 - z.kang().centroid, 1e-6))
    return g


def z_strength(g, n):
    """Sum of edge A (defuzzified) and the fused opinion of those edges."""
    strength, op = 0.0, None
    for _, _, data in g.edges(n, data=True):
        strength += data["z"].A.centroid
        op = data["op"] if op is None else fuse(op, data["op"])
    return strength, op


def most_reliable_path(g, src, dst):
    """Path maximising product of beliefs (Dijkstra on -log b) and its discounted opinion."""
    path = nx.dijkstra_path(g, src, dst, weight="rel_cost")
    op = None
    for a, b in zip(path, path[1:]):
        e = g.edges[a, b]["op"]
        op = e if op is None else discount(op, e)
    return path, op


def z_closeness(g):
    """Harmonic closeness on distance 1 - Kang-defuzzified edge Z."""
    return nx.harmonic_centrality(g, distance="dist")


def fragile_core(g, central_q=0.75, reliable_q=0.25):
    """Nodes that are central (top strength quantile) but weakly held (bottom reliability quantile)."""
    if g.number_of_nodes() == 0:
        return []
    strength = {n: z_strength(g, n)[0] for n in g}
    rel = {n: g.nodes[n]["op"].projected for n in g}

    def q(vals, p):
        v = sorted(vals)
        return v[min(int(p * len(v)), len(v) - 1)]

    s_cut, r_cut = q(strength.values(), central_q), q(rel.values(), reliable_q)
    return [n for n in g if strength[n] >= s_cut and rel[n] <= r_cut]


In [ ]:
%%writefile zfmn/evidence.py
"""Pool a target text and its resampled siblings into evidence, opinions and Z-numbers,
and map human hallucination spans onto the target's nodes and edges."""

import math

from .zops import ZNumber, from_evidence, reliability_tfn, tfn_from_values


def _overlaps(span, spans):
    s, e = span
    return any(s < le and e > ls for ls, le in spans)


def _occ_conf(occs, kind):
    vals = []
    for o in occs:
        if kind == "node":
            if o["stats"] is not None:
                vals.append(o["stats"]["conf"])
        elif o["path_conf"] is not None:
            vals.append(o["path_conf"])
    return sum(vals) / len(vals) if vals else None


def element_records(target, samples, spans, W=2.0, a=0.5):
    """One record per node and edge of the target TFMN.

    target, samples: TFMN dicts from network.build_tfmn (built with token stats).
    spans: human hallucination spans [(start, end)] in the target text.

    Evidence over T = 1 + len(samples) texts: r = sum of the element's confidence in each
    text where it appears (0 where absent), s = T - r.
    """
    T = 1 + len(samples)
    recs = []
    for kind, key_name in (("node", "nodes"), ("edge", "edges")):
        for key, occs in target[key_name].items():
            per_text = [_occ_conf(occs, kind)]
            present = 1
            for smp in samples:
                o = smp[key_name].get(key)
                per_text.append(_occ_conf(o, kind) if o else None)
                present += 1 if o else 0
            confs = [c if c is not None else 0.0 for c in per_text]
            r = sum(confs)
            op = from_evidence(r, T - r, W=W, a=a)
            op_freq = from_evidence(present, T - present, W=W, a=a)  # beta-only baseline
            if kind == "node":
                stats = [o["stats"] for o in occs if o["stats"] is not None]
                wrong = any(_overlaps(o["span"], spans) for o in occs)
                mean_lp = sum(s["mean_lp"] for s in stats) / len(stats) if stats else None
                min_lp = min((s["min_lp"] for s in stats), default=None)
                ent = sum(s["entropy"] for s in stats) / len(stats) if stats else None
            else:
                wrong = any(_overlaps(sp, spans) for o in occs for sp in o["spans"])
                pc = [o["path_conf"] for o in occs if o["path_conf"] is not None]
                mean_lp = min_lp = ent = None
                if pc:
                    mean_lp = sum(math.log(max(c, 1e-12)) for c in pc) / len(pc)
            # A: the element's confidence across texts (0 where absent) as (min, median, max)
            A = tfn_from_values(confs)
            z = ZNumber(A, reliability_tfn(op))
            recs.append({
                "kind": kind, "key": key if kind == "node" else "|".join(key),
                "wrong": int(wrong), "n_occ": len(occs),
                "target_conf": per_text[0], "mean_lp": mean_lp, "min_lp": min_lp,
                "entropy": ent, "freq": present / T,
                "sl_b": op.b, "sl_u": op.u, "sl_P": op.projected,
                "freq_P": op_freq.projected, "z_kang": z.kang().centroid,
            })
    return recs


# Scores where higher means "more likely hallucinated". None-safe.
SCORERS = {
    "neg_mean_logprob": lambda r: None if r["mean_lp"] is None else -r["mean_lp"],
    "neg_min_logprob": lambda r: None if r["min_lp"] is None else -r["min_lp"],
    "entropy": lambda r: r["entropy"],
    "beta_frequency_only": lambda r: 1 - r["freq_P"],
    "subjective_logic_P": lambda r: 1 - r["sl_P"],
    "z_fmn_hybrid": lambda r: 1 - r["z_kang"],
}


In [ ]:
%%writefile zfmn/advanced.py
"""Network-aware evidence and propagation for Z-FMN.

Three additions on top of evidence.py:

1. Context-conditioned evidence. A resample that does not mention a concept is not automatically
   evidence against it. If the resample talks about the same context (the concept's network
   neighbours) but leaves the concept out, that is disbelief. If it never touches that context,
   it is silence, which adds to uncertainty only. Subjective logic keeps the two apart; plain
   frequency cannot.
2. Relational propagation. A concept's reliability is fused with the support it receives from
   its neighbours through the links that connect them (trust discounting along each link,
   averaging fusion across links). Hallucinated material tends to form small, weakly supported
   clusters, which this exposes.
3. Emotional value. The fuzzy value A of a concept is the valence of its semantic frame (the
   mean EmoAtlas valence of its neighbours) across the answer and its resamples.
"""

from collections import defaultdict

from .zops import Opinion, TFN, ZNumber, discount, from_evidence, reliability_tfn, tfn_from_values

# ---- lexicons --------------------------------------------------------------------------------

_VALENCE = None
_EMOTIONS = None


def load_emoatlas_lexicons():
    """Valence and Plutchik emotion lexicons shipped with EmoAtlas (pip install emoatlas --no-deps)."""
    global _VALENCE, _EMOTIONS
    if _VALENCE is None:
        import json
        import os
        import runpy
        import importlib.util

        root = importlib.util.find_spec("emoatlas").submodule_search_locations[0]
        v = runpy.run_path(os.path.join(root, "valence", "english.py"))
        _VALENCE = {w: 1.0 for w in v["_positive"]}
        _VALENCE.update({w: -1.0 for w in v["_negative"]})
        with open(os.path.join(root, "langs", "english.json")) as f:
            _EMOTIONS = json.load(f)
    return _VALENCE, _EMOTIONS


def valence(word):
    val, _ = load_emoatlas_lexicons()
    return val.get(word, 0.0)


def emotions(word):
    _, emo = load_emoatlas_lexicons()
    return emo.get(word, [])


# ---- helpers ---------------------------------------------------------------------------------


def _conf(occs, kind):
    vals = []
    for o in occs:
        c = o["stats"]["conf"] if kind == "node" and o["stats"] else o.get("path_conf")
        if c is not None:
            vals.append(c)
    return sum(vals) / len(vals) if vals else 0.0


def neighbours(net):
    nb = defaultdict(set)
    for a, b in net["edges"]:
        nb[a].add(b)
        nb[b].add(a)
    return nb


def average_fuse(ops):
    """Averaging fusion (Josang) of several opinions: for dependent sources such as links."""
    ops = [o for o in ops if o is not None]
    if not ops:
        return None
    if any(o.u == 0 for o in ops):
        dog = [o for o in ops if o.u == 0]
        b = sum(o.b for o in dog) / len(dog)
        return Opinion(b, 1 - b, 0.0, dog[0].a)
    inv = sum(1 / o.u for o in ops)
    b = sum(o.b / o.u for o in ops) / inv
    u = len(ops) / inv
    return Opinion(b, 1 - b - u, u, sum(o.a for o in ops) / len(ops))


def _overlaps(span, spans):
    s, e = span
    return any(s < le and e > ls for ls, le in spans)


# ---- context-conditioned opinions --------------------------------------------------------------


def basic_opinions(target, samples, W=2.0, a=0.5):
    """Opinions where every resample that omits the element counts against it."""
    T = 1 + len(samples)
    node_ops, edge_ops = {}, {}
    for key_name, kind, ops in (("nodes", "node", node_ops), ("edges", "edge", edge_ops)):
        for k, occs in target[key_name].items():
            r = _conf(occs, kind) + sum(
                _conf(s[key_name][k], kind) for s in samples if k in s[key_name])
            ops[k] = from_evidence(r, T - r, W=W, a=a)
    return node_ops, edge_ops


def context_opinions(target, samples, W=2.0, a=0.5):
    """Opinions for every node and edge of the target using context-conditioned evidence."""
    nb = neighbours(target)
    node_ops, edge_ops = {}, {}
    for v, occs in target["nodes"].items():
        r, s = _conf(occs, "node"), 0.0
        ctx = nb.get(v, set())
        for smp in samples:
            if v in smp["nodes"]:
                r += _conf(smp["nodes"][v], "node")
            elif ctx:
                s += sum(1 for u in ctx if u in smp["nodes"]) / len(ctx)
            # no context at all: silence, adds nothing
        node_ops[v] = from_evidence(r, s, W=W, a=a)
    for (x, y), occs in target["edges"].items():
        r, s = _conf(occs, "edge"), 0.0
        for smp in samples:
            if (x, y) in smp["edges"]:
                r += _conf(smp["edges"][(x, y)], "edge")
            else:
                s += 0.5 * ((x in smp["nodes"]) + (y in smp["nodes"]))
        edge_ops[(x, y)] = from_evidence(r, s, W=W, a=a)
    return node_ops, edge_ops


def propagate(node_ops, edge_ops, nb):
    """Fuse each node's own opinion with relational support from its neighbours.

    Support from neighbour u through link e = discount(op(e), op(u)): u vouches for v only as far
    as the link between them is itself reliable. Link supports are combined by averaging fusion,
    then fused with the node's own opinion. Edges get the discounted support of both endpoints.
    """
    new_nodes = {}
    for v, own in node_ops.items():
        sup = []
        for u in nb.get(v, ()):
            e = edge_ops.get((v, u) if v < u else (u, v))
            if e is not None and u in node_ops:
                sup.append(discount(e, node_ops[u]))
        support = average_fuse(sup)
        new_nodes[v] = own if support is None else average_fuse([own, support])
    new_edges = {}
    for (x, y), own in edge_ops.items():
        ends = average_fuse([node_ops.get(x), node_ops.get(y)])
        new_edges[(x, y)] = own if ends is None else average_fuse([own, discount(ends, own)])
    return new_nodes, new_edges


def source_opinions(target, source, W=2.0, a=0.5, strength=2.0):
    """External evidence from the source network (the document the model was given).

    Node in the source: belief. Node absent from the source: disbelief.
    Edge in the source network: belief. Both endpoints in the source but never linked there:
    disbelief (the answer relates two source concepts in a way the source does not). Endpoints
    missing from the source: silence (uncertainty), already covered by the node opinions.
    `strength` is how many resamples one source observation is worth.
    """
    node_ops, edge_ops = {}, {}
    for v in target["nodes"]:
        hit = v in source["nodes"]
        node_ops[v] = from_evidence(strength * hit, strength * (not hit), W=W, a=a)
    for (x, y) in target["edges"]:
        if (x, y) in source["edges"]:
            r, s = strength, 0.0
        elif x in source["nodes"] and y in source["nodes"]:
            r, s = 0.0, strength
        else:
            r, s = 0.0, 0.0
        edge_ops[(x, y)] = from_evidence(r, s, W=W, a=a)
    return node_ops, edge_ops


# ---- records -----------------------------------------------------------------------------------


def _src_fields(src_op, model_op, ctx_op):
    """Source opinion alone and fused with the model's own opinion (cumulative fusion)."""
    from .zops import fuse

    if src_op is None:
        return {}
    f = fuse(model_op, src_op)
    fc = fuse(ctx_op, src_op)
    return {"src_P": src_op.projected, "src_d": src_op.d, "src_u": src_op.u,
            "fused_P": f.projected, "fused_d": f.d, "fused_u": f.u,
            "fused_ctx_P": fc.projected, "fused_ctx_d": fc.d, "fused_ctx_u": fc.u}


def frame_valence(net, v, nb):
    vals = [valence(u) for u in nb.get(v, ())]
    return sum(vals) / len(vals) if vals else 0.0


def _types(spans_list, spans, span_types):
    """RAGTruth hallucination types touching any of the given char spans."""
    if not span_types:
        return []
    return sorted({t for (ls, le), t in zip(spans, span_types)
                   for s, e in spans_list if s < le and e > ls})


def advanced_records(target, samples, spans, W=2.0, a=0.5, span_types=None, source=None):
    nb_t = neighbours(target)
    node_ops, edge_ops = context_opinions(target, samples, W, a)
    p_nodes, p_edges = propagate(node_ops, edge_ops, nb_t)
    b_nodes, b_edges = basic_opinions(target, samples, W, a)
    pb_nodes, pb_edges = propagate(b_nodes, b_edges, nb_t)
    if source is not None:
        s_nodes, s_edges = source_opinions(target, source, W, a)
    else:
        s_nodes = s_edges = {}
    nbs = [neighbours(s) for s in samples]
    recs = []
    for v, occs in target["nodes"].items():
        # A: frame valence across the answer and the resamples that mention v
        fv = [frame_valence(target, v, nb_t)] + [
            frame_valence(s, v, nb) for s, nb in zip(samples, nbs) if v in s["nodes"]]
        A = tfn_from_values(fv)
        z = ZNumber(A, reliability_tfn(p_nodes[v]))
        recs.append({
            "kind": "node", "key": v, "wrong": int(any(_overlaps(o["span"], spans) for o in occs)),
            "types": _types([o["span"] for o in occs], spans, span_types),
            "ctx_P": node_ops[v].projected, "ctx_u": node_ops[v].u, "ctx_d": node_ops[v].d,
            "ctx_b": node_ops[v].b, "prop_P": p_nodes[v].projected,
            "basic_prop_P": pb_nodes[v].projected,
            **_src_fields(s_nodes.get(v), b_nodes[v], node_ops[v]),
            "valence": valence(v), "n_emotions": len(emotions(v)),
            "frame_valence": A.centroid, "frame_valence_spread": A.r - A.l,
            "degree": len(nb_t.get(v, ())),
            "z": [A.l, A.m, A.r, z.B.l, z.B.m, z.B.r],
        })
    for (x, y), occs in target["edges"].items():
        recs.append({
            "kind": "edge", "key": f"{x}|{y}",
            "wrong": int(any(_overlaps(sp, spans) for o in occs for sp in o["spans"])),
            "types": _types([sp for o in occs for sp in o["spans"]], spans, span_types),
            "ctx_P": edge_ops[(x, y)].projected, "ctx_u": edge_ops[(x, y)].u,
            "ctx_d": edge_ops[(x, y)].d, "ctx_b": edge_ops[(x, y)].b,
            "prop_P": p_edges[(x, y)].projected, "basic_prop_P": pb_edges[(x, y)].projected,
            **_src_fields(s_edges.get((x, y)), b_edges[(x, y)], edge_ops[(x, y)]),
        })
    return recs


ADVANCED_SCORERS = {
    "context_conditioned_P": lambda r: 1 - r["ctx_P"],
    "network_propagated_P": lambda r: 1 - r["prop_P"],
}


In [ ]:
%%writefile zfmn/metrics.py
"""AUROC with a bootstrap that resamples whole responses (clustered)."""

import random


def auroc(scores, labels):
    """Mann-Whitney AUROC with average ranks for ties. None if one class is missing."""
    pairs = sorted(zip(scores, labels))
    n_pos = sum(labels)
    n_neg = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    rank_sum, i = 0.0, 0
    while i < len(pairs):
        j = i
        while j < len(pairs) and pairs[j][0] == pairs[i][0]:
            j += 1
        avg_rank = (i + j + 1) / 2  # ranks are 1-based
        rank_sum += avg_rank * sum(l for _, l in pairs[i:j])
        i = j
    return (rank_sum - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def clustered_auroc(groups, n_boot=1000, seed=0):
    """groups: list of (scores, labels) per response. Returns (auroc, ci_low, ci_high)."""
    flat_s = [s for g in groups for s in g[0]]
    flat_l = [l for g in groups for l in g[1]]
    point = auroc(flat_s, flat_l)
    rng = random.Random(seed)
    boots = []
    for _ in range(n_boot):
        pick = [groups[rng.randrange(len(groups))] for _ in groups]
        a = auroc([s for g in pick for s in g[0]], [l for g in pick for l in g[1]])
        if a is not None:
            boots.append(a)
    boots.sort()
    if not boots:
        return point, None, None
    return point, boots[int(0.025 * len(boots))], boots[min(int(0.975 * len(boots)), len(boots) - 1)]


In [ ]:
%%writefile zfmn/grounding.py
"""Sentence-level natural-language inference (NLI) evidence for Z-FMN.

Two uses of one NLI model:

1. Source grounding. Each answer sentence is checked against the source the model was given
   (SummaC-ZS style: the source is split into chunks, and the chunk with the highest
   entailment-minus-contradiction score decides). Entailment is belief, contradiction is
   disbelief, neutral is missing evidence. This is the external evidence that model-internal
   confidence lacks, and the one that can catch confidently held contradictions.
2. SelfCheckGPT-NLI baseline (Manakul et al., 2023). Each answer sentence is checked against
   every resample; the score is the mean of P(contradiction) / (P(contradiction) + P(entailment)).

Sentence scores are mapped onto the TFMN elements whose words lie in that sentence. When an
element occurs in several sentences, its most contradicted occurrence counts.
"""

import json

from .zops import Opinion, from_evidence, fuse


def flatten_source(text):
    """Turn a JSON source (data-to-text) into 'key: value.' lines that an NLI model can read."""
    try:
        obj = json.loads(text)
    except (ValueError, TypeError):
        return text
    lines = []

    def walk(o, prefix=""):
        if isinstance(o, dict):
            for k, v in o.items():
                walk(v, f"{prefix}{k} ")
        elif isinstance(o, list):
            for v in o:
                walk(v, prefix)
        elif o not in (None, ""):
            lines.append(f"{prefix.strip()}: {o}.")

    walk(obj)
    return "\n".join(lines) if lines else text


def sentences(text, nlp):
    """Character spans of the sentences in text."""
    return [(s.start_char, s.end_char) for s in nlp(text).sents if s.text.strip()]


def chunk_text(text, nlp, max_words=300):
    """Consecutive sentence windows of at most max_words words, overlapping by one sentence."""
    sents = [s.text.strip() for s in nlp(text).sents if s.text.strip()]
    if not sents:
        return [text]
    chunks, cur, n = [], [], 0
    for s in sents:
        w = len(s.split())
        if cur and n + w > max_words:
            chunks.append(" ".join(cur))
            cur, n = cur[-1:], len(cur[-1].split())
        cur.append(s)
        n += w
    chunks.append(" ".join(cur))
    return chunks


class NLI:
    """Batched NLI with any Hugging Face MNLI classifier. Returns (entail, neutral, contra) per pair."""

    def __init__(self, name="microsoft/deberta-large-mnli", device=None, batch_size=16, max_length=512):
        import torch
        from transformers import AutoModelForSequenceClassification, AutoTokenizer

        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tok = AutoTokenizer.from_pretrained(name)
        # float32: DeBERTa's attention fails in half precision ("expected scalar type Float but found Half")
        self.model = AutoModelForSequenceClassification.from_pretrained(name, torch_dtype=torch.float32).to(self.device).eval()
        labels = {i: l.lower() for i, l in self.model.config.id2label.items()}
        self.idx = {k: next(i for i, l in labels.items() if l.startswith(k)) for k in ("entail", "neutral", "contra")}
        self.batch_size, self.max_length = batch_size, max_length

    def __call__(self, pairs):
        out = []
        with self.torch.no_grad():
            for i in range(0, len(pairs), self.batch_size):
                prem, hyp = zip(*pairs[i:i + self.batch_size])
                # truncate the premise only, so the hypothesis is always read in full
                enc = self.tok(list(prem), list(hyp), truncation="only_first", max_length=self.max_length,
                               padding=True, return_tensors="pt").to(self.device)
                p = self.model(**enc).logits.float().softmax(-1).tolist()
                out += [(r[self.idx["entail"]], r[self.idx["neutral"]], r[self.idx["contra"]]) for r in p]
        return out


def nli_record(response, source, samples, nlp, nli):
    """Everything NLI says about one answer: per-sentence source grounding and SelfCheckGPT scores."""
    sents = sentences(response, nlp)
    hyps = [response[s:e] for s, e in sents]
    chunks = chunk_text(flatten_source(source), nlp)
    sample_texts = [s for s in samples if s.strip()]
    pairs = [(c, h) for h in hyps for c in chunks] + [(t, h) for h in hyps for t in sample_texts]
    probs = nli(pairs) if pairs else []
    nc, ns = len(chunks), len(sample_texts)
    src, selfcheck = [], []
    for i in range(len(hyps)):
        per_chunk = probs[i * nc:(i + 1) * nc]
        e, n, c = max(per_chunk, key=lambda p: p[0] - p[2])
        src.append([e, n, c, max(p[2] for p in per_chunk)])
        off = len(hyps) * nc + i * ns
        per_sample = probs[off:off + ns]
        selfcheck.append(sum(c / (c + e + 1e-9) for e, _, c in per_sample) / ns if ns else None)
    return {"sents": sents, "src": src, "selfcheck": selfcheck}


def _sentence_of(span, sents):
    for i, (s, e) in enumerate(sents):
        if span[0] < e and span[1] > s:
            return i
    return None


def element_nli(occ_spans, rec):
    """NLI fields for one element, given the char spans of its occurrences in the answer."""
    idx = {_sentence_of(sp, rec["sents"]) for sp in occ_spans} - {None}
    if not idx:
        return None
    i = max(idx, key=lambda j: rec["src"][j][2])  # most contradicted occurrence
    e, n, c, cmax = rec["src"][i]
    sc = [rec["selfcheck"][j] for j in idx if rec["selfcheck"][j] is not None]
    return {"nli_e": e, "nli_n": n, "nli_c": c, "nli_cmax": cmax,
            "selfcheck": max(sc) if sc else None}


def nli_opinion(e, c, strength=10.0, W=2.0, a=0.5):
    """NLI probabilities as subjective-logic evidence worth `strength` observations.

    Entailment is positive evidence, contradiction negative, and neutral is no evidence, so a
    neutral verdict leaves the opinion uncertain instead of pushing it either way.
    """
    return from_evidence(strength * e, strength * c, W=W, a=a)


def fused_fields(sl_b, sl_u, e, c, strength=10.0, W=2.0, a=0.5, selfcheck=None, n_samples=10):
    """Model opinion fused with source entailment, and optionally with sample consistency.

    The three-way fusion adds SelfCheckGPT's sentence-level verdict as a third opinion worth
    n_samples observations: consistent resamples are belief, contradicting ones disbelief. It is
    semantic agreement with the samples, which the word-level model opinion does not measure.
    """
    model = Opinion(sl_b, max(0.0, 1 - sl_b - sl_u), sl_u, a)
    f = fuse(model, nli_opinion(e, c, strength, W, a))
    out = {"nli_fused_P": f.projected, "nli_fused_d": f.d, "nli_fused_u": f.u}
    if selfcheck is not None:
        f3 = fuse(f, from_evidence(n_samples * (1 - selfcheck), n_samples * selfcheck, W=W, a=a))
        out.update({"tri_fused_P": f3.projected, "tri_fused_d": f3.d, "tri_fused_u": f3.u})
    return out


In [ ]:
%%writefile tests/test_core.py
"""Checks that need no language model: python -m pytest tests (or python tests/test_core.py)."""

import os
import sys

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), ".."))

from zfmn.metrics import auroc  # noqa: E402
from zfmn.network import build_tfmn  # noqa: E402
from zfmn.zops import Opinion, discount, from_evidence, fuse  # noqa: E402


def close(a, b, tol=1e-9):
    return abs(a - b) < tol


def test_opinions_sum_to_one():
    o1, o2 = from_evidence(8, 2), from_evidence(3, 5)
    for o in (o1, o2, discount(o1, o2), fuse(o1, o2)):
        assert close(o.b + o.d + o.u, 1.0)
    # discounting can only lower belief; fusion lowers uncertainty
    assert discount(o1, o2).b <= min(o1.b, o2.b)
    assert fuse(o1, o2).u < min(o1.u, o2.u)


def test_fusion_matches_pooled_evidence():
    # cumulative fusion of two evidence-based opinions equals the opinion from pooled evidence
    f = fuse(from_evidence(4, 1), from_evidence(2, 3))
    p = from_evidence(6, 4)
    assert close(f.b, p.b) and close(f.u, p.u)


def test_auroc():
    assert auroc([0.1, 0.2, 0.8, 0.9], [0, 0, 1, 1]) == 1.0
    assert auroc([0.9, 0.8, 0.2, 0.1], [0, 0, 1, 1]) == 0.0
    assert auroc([0.5, 0.5, 0.5, 0.5], [0, 1, 0, 1]) == 0.5


def test_alignment_and_edges():
    text = "Photosynthesis converts light energy into chemical energy."
    # fake LM tokens: split every word in two halves with known log-probs
    toks, i = [], 0
    for w in text.split(" "):
        s = text.index(w, i)
        h = s + len(w) // 2
        toks += [{"start": s, "end": h, "logprob": -1.0, "entropy": 1.0},
                 {"start": h, "end": s + len(w), "logprob": -3.0, "entropy": 2.0}]
        i = s + len(w)
    net = build_tfmn(text, toks)
    assert "photosynthesis" in net["nodes"] and "light" in net["nodes"]
    st = net["nodes"]["photosynthesis"][0]["stats"]
    assert close(st["joint_lp"], -4.0) and close(st["mean_lp"], -2.0) and close(st["min_lp"], -3.0)
    assert any("photosynthesis" in k for k in net["edges"])
    for occs in net["edges"].values():
        assert all(o["path_conf"] is not None for o in occs)


def test_grounding():
    from zfmn.grounding import element_nli, flatten_source, fused_fields, nli_record
    from zfmn.network import get_nlp

    def fake_nli(pairs):  # entailed if the hypothesis words all occur in the premise, else contradicted
        out = []
        for prem, hyp in pairs:
            ok = all(w.strip(".").lower() in prem.lower() for w in hyp.split())
            out.append((0.9, 0.05, 0.05) if ok else (0.05, 0.05, 0.9))
        return out

    src = "The bridge opened in 1932. It is made of steel."
    resp = "The bridge opened in 1932. It is made of wood."
    rec = nli_record(resp, src, [src, "The bridge opened in 1932."], get_nlp(), fake_nli)
    assert len(rec["sents"]) == 2
    assert rec["src"][0][0] > 0.5 and rec["src"][1][2] > 0.5  # second sentence contradicted
    wood = resp.index("wood")
    en = element_nli([(wood, wood + 4)], rec)
    assert en["nli_c"] > 0.5 and en["selfcheck"] > 0.5
    f = fused_fields(0.8, 0.1, en["nli_e"], en["nli_c"])
    assert f["nli_fused_d"] > 0.3 and f["nli_fused_P"] < 0.65  # contradiction pulls a confident element down
    assert "name: Joe's." in flatten_source('{"name": "Joe\'s", "tags": []}')


if __name__ == "__main__":
    for name, fn in list(globals().items()):
        if name.startswith("test_"):
            fn()
            print("ok", name)


In [ ]:
%%writefile run_pilot.py
"""Z-FMN on RAGTruth.

For each response written by an open model:
  1. score it with that same model (token log-probs and entropies, teacher forcing),
  2. draw N fresh samples for the same prompt and score them too,
  3. check every answer sentence with an NLI model against the source and against the samples,
  4. build TFMNs, pool evidence into hybrid Z-numbers,
  5. map RAGTruth's human hallucination spans onto nodes and edges,
then compare how well each confidence measure detects hallucinated nodes, edges and answers (AUROC).

Stages (--stage): generate (needs the LLM), nli (needs the NLI model), evaluate (CPU), or all.
Each stage is cached in --out, and --reuse-dir picks up generations and NLI results from earlier runs (for
example a previous Kaggle version attached as input), so a long study can be split across sessions.

Example (Kaggle / Colab GPU):
  python run_pilot.py --model-label mistral-7B-instruct --limit 50 --n-samples 10 --load-in-4bit
"""

import argparse
import csv
import glob
import itertools
import json
import os
import sys
import time

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

from zfmn.data import OPEN_MODELS, load_ragtruth  # noqa: E402


def read_jsonl(path):
    if not os.path.exists(path):
        return {}
    with open(path) as f:
        return {d["id"]: d for d in map(json.loads, f)}


def reuse_cached(args, rows, path, ok=lambda d: True):
    """Copy records for these rows from same-named files under --reuse-dir into path."""
    if not args.reuse_dir:
        return
    name = os.path.basename(path)
    want = {r["id"] for r in rows} - set(read_jsonl(path))
    found = {}
    for p in glob.glob(os.path.join(args.reuse_dir, "**", name), recursive=True):
        if os.path.abspath(p) == os.path.abspath(path):
            continue
        for k, d in read_jsonl(p).items():
            if k in want and k not in found and ok(d):
                found[k] = d
    if found:
        with open(path, "a") as f:
            for d in found.values():
                f.write(json.dumps(d) + "\n")
    print(f"[reuse] {len(found)} earlier records reused from {name} files", flush=True)


def stage_generate(args, rows, path):
    from zfmn.lm import load_model, sample_responses, score_response

    reuse_cached(args, rows, path, ok=lambda d: len(d["samples"]) == args.n_samples)
    done = read_jsonl(path)
    todo = [r for r in rows if r["id"] not in done]
    if not todo:
        return
    t0 = time.time()
    model, tok = load_model(args.hf_model, load_in_4bit=args.load_in_4bit)
    with open(path, "a") as f:
        for i, r in enumerate(todo, 1):
            if args.time_budget_hours and time.time() - t0 > args.time_budget_hours * 3600:
                print(f"[generate] time budget reached; {len(todo) - i + 1} left for the next session", flush=True)
                break
            target = score_response(model, tok, r["prompt"], r["response"], args.max_length)
            texts = sample_responses(model, tok, r["prompt"], args.n_samples,
                                     args.temperature, args.max_new_tokens)
            samples = [{"text": t, "tokens": score_response(model, tok, r["prompt"], t, args.max_length)}
                       for t in texts]
            f.write(json.dumps({"id": r["id"], "target_tokens": target, "samples": samples}) + "\n")
            f.flush()
            print(f"[generate] {i}/{len(todo)} response {r['id']}", flush=True)


def stage_nli(args, rows, gen_path, nli_path):
    from zfmn.grounding import NLI, nli_record
    from zfmn.network import get_nlp

    gen = read_jsonl(gen_path)
    reuse_cached(args, [r for r in rows if r["id"] in gen], nli_path)
    done = read_jsonl(nli_path)
    todo = [r for r in rows if r["id"] in gen and r["id"] not in done]
    if not todo:
        return
    nli, nlp = NLI(args.nli_model, batch_size=args.nli_batch), get_nlp()
    with open(nli_path, "a") as f:
        for i, r in enumerate(todo, 1):
            rec = nli_record(r["response"], r["source_text"], [s["text"] for s in gen[r["id"]]["samples"]], nlp, nli)
            f.write(json.dumps({"id": r["id"], **rec}) + "\n")
            f.flush()
            if i % 10 == 0 or i == len(todo):
                print(f"[nli] {i}/{len(todo)}", flush=True)


def response_record(r, g, recs, nli):
    """One row per answer: does it contain any hallucination, and answer-level scores."""
    lps = [t["logprob"] for t in g["target_tokens"]]
    out = {"response_id": r["id"], "task_type": r["task_type"], "wrong": int(bool(r["spans"])),
           "types": ";".join(sorted(set(r["span_types"]))), "n_tokens": len(lps),
           "resp_neg_mean_logprob": -sum(lps) / len(lps) if lps else None}
    if nli:
        sc = [s for s in nli["selfcheck"] if s is not None]
        out["resp_selfcheck_nli"] = sum(sc) / len(sc) if sc else None
        out["resp_nli_contra_max"] = max((s[2] for s in nli["src"]), default=None)
        out["resp_nli_not_entailed"] = 1 - min((s[0] for s in nli["src"]), default=1.0)
    for kind in ("node", "edge"):
        xs = [x for x in recs if x["kind"] == kind]
        for name, f in (("P", lambda x: 1 - x["sl_P"]), ("u", lambda x: x.get("ctx_u")),
                        ("nli_fused_P", lambda x: None if x.get("nli_fused_P") is None else 1 - x["nli_fused_P"]),
                        ("tri_fused_P", lambda x: None if x.get("tri_fused_P") is None else 1 - x["tri_fused_P"])):
            v = [f(x) for x in xs if f(x) is not None]
            out[f"resp_{kind}_{name}_max"] = max(v) if v else None
            out[f"resp_{kind}_{name}_mean"] = sum(v) / len(v) if v else None
    return out


def stage_evaluate(args, rows, gen_path, nli_path):
    from zfmn.advanced import advanced_records
    from zfmn.evidence import SCORERS, element_records
    from zfmn.grounding import element_nli, fused_fields
    from zfmn.metrics import clustered_auroc
    from zfmn.network import build_tfmn

    scorers = dict(SCORERS)
    scorers.update({
        "uncertainty_u (unsupported)": lambda x: x["ctx_u"],
        "disbelief_d (contradicted)": lambda x: x["ctx_d"],
        "source_network_only": lambda x: 1 - x["src_P"],
        "fused_model+source": lambda x: 1 - x["fused_P"],
        "selfcheck_nli (baseline)": lambda x: x.get("selfcheck"),
        "nli_source_contradiction": lambda x: x.get("nli_c"),
        "nli_source_not_entailed": lambda x: None if x.get("nli_e") is None else 1 - x["nli_e"],
        "z_fmn+nli_fused_P": lambda x: None if x.get("nli_fused_P") is None else 1 - x["nli_fused_P"],
        "z_fmn+nli_fused_d": lambda x: x.get("nli_fused_d"),
        "z_fmn+nli+selfcheck_P": lambda x: None if x.get("tri_fused_P") is None else 1 - x["tri_fused_P"],
    })

    gen = read_jsonl(gen_path)
    nli_all = read_jsonl(nli_path)
    all_recs, resp_recs = [], []
    for r in rows:
        g = gen.get(r["id"])
        if g is None:
            continue
        target = build_tfmn(r["response"], g["target_tokens"], args.max_distance)
        samples = [build_tfmn(s["text"], s["tokens"], args.max_distance) for s in g["samples"]]
        source = build_tfmn(r["source_text"], None, args.max_distance)
        adv = advanced_records(target, samples, r["spans"], W=args.prior_weight,
                               span_types=r["span_types"], source=source)
        nli = nli_all.get(r["id"])
        recs = []
        for rec, extra in zip(element_records(target, samples, r["spans"], W=args.prior_weight), adv):
            extra = {k: v for k, v in extra.items() if k not in ("kind", "key", "wrong")}
            extra["types"] = ";".join(extra["types"])
            extra.pop("z", None)
            rec.update(extra)
            if nli:
                if rec["kind"] == "node":
                    spans = [o["span"] for o in target["nodes"][rec["key"]]]
                else:
                    spans = [sp for o in target["edges"][tuple(rec["key"].split("|"))] for sp in o["spans"]]
                en = element_nli(spans, nli)
                if en:
                    rec.update(en)
                    rec.update(fused_fields(rec["sl_b"], rec["sl_u"], en["nli_e"], en["nli_c"],
                                            args.nli_strength, args.prior_weight,
                                            selfcheck=en["selfcheck"], n_samples=args.n_samples))
            rec["response_id"] = r["id"]
            rec["task_type"] = r["task_type"]
            recs.append(rec)
        all_recs += recs
        resp_recs.append(response_record(r, g, recs, nli))

    os.makedirs(args.out, exist_ok=True)
    for name, data in (("elements.csv", all_recs), ("responses.csv", resp_recs)):
        keys = list(dict.fromkeys(k for d in data for k in d))
        with open(os.path.join(args.out, name), "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=keys)
            w.writeheader()
            w.writerows(data)

    summary = []
    fmt = lambda v: "n/a" if v is None else f"{v:.3f}"  # noqa: E731
    subsets = {
        "all": lambda x: True,
        "conflict": lambda x: not x["wrong"] or "Conflict" in x["types"],
        "baseless": lambda x: not x["wrong"] or "Baseless" in x["types"],
    }
    lines = ["| Element | Hallucination type | Measure | AUROC | 95% CI | n (wrong / total) |",
             "| --- | --- | --- | --- | --- | --- |"]
    for kind, (sub, keep) in [(k, kv) for k in ("node", "edge") for kv in subsets.items()]:
        recs = [x for x in all_recs if x["kind"] == kind and keep(x)]
        for name, fn in scorers.items():
            groups = {}
            for x in recs:
                try:
                    s = fn(x)
                except KeyError:
                    s = None
                if s is None:
                    continue
                gs, gl = groups.setdefault(x["response_id"], ([], []))
                gs.append(s)
                gl.append(x["wrong"])
            groups = list(groups.values())
            n = sum(len(g[1]) for g in groups)
            pos = sum(sum(g[1]) for g in groups)
            if not n or pos in (0, n):
                continue
            a, lo, hi = clustered_auroc(groups, n_boot=args.n_boot)
            lines.append(f"| {kind} | {sub} | {name} | {fmt(a)} | [{fmt(lo)}, {fmt(hi)}] | {pos} / {n} |")
            summary.append({"kind": kind, "subset": sub, "measure": name, "auroc": a, "ci_low": lo, "ci_high": hi,
                            "n_wrong": pos, "n": n})
    # answer level: does the answer contain any hallucination? Needs answers without hallucinations too.
    labels = [x["wrong"] for x in resp_recs]
    if 0 < sum(labels) < len(labels):
        lines += ["", "| Answer-level measure | AUROC | 95% CI | n (with hallucination / total) |",
                  "| --- | --- | --- | --- |"]
        for k in [k for k in resp_recs[0] if k.startswith("resp_")]:
            pairs = [(x[k], x["wrong"]) for x in resp_recs if x.get(k) is not None]
            if len({lab for _, lab in pairs}) < 2:
                continue
            a, lo, hi = clustered_auroc([([s], [lab]) for s, lab in pairs], n_boot=args.n_boot)
            npos = sum(lab for _, lab in pairs)
            lines.append(f"| {k} | {fmt(a)} | [{fmt(lo)}, {fmt(hi)}] | {npos} / {len(pairs)} |")
            summary.append({"kind": "answer", "subset": "all", "measure": k, "auroc": a, "ci_low": lo,
                            "ci_high": hi, "n_wrong": npos, "n": len(pairs)})
    table = "\n".join(lines)
    with open(os.path.join(args.out, "results.md"), "w") as f:
        f.write(f"# Z-FMN on RAGTruth\n\nModel: {args.hf_model}; tasks: {args.tasks}; answers: {len(resp_recs)}; "
                f"samples per prompt: {args.n_samples}; NLI: {args.nli_model if nli_all else 'not run'}\n\n")
        f.write(table + "\n")
    with open(os.path.join(args.out, "results.json"), "w") as f:
        json.dump(summary, f, indent=1)
    print(table)


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--model-label", default="mistral-7B-instruct", choices=sorted(OPEN_MODELS),
                   help="RAGTruth generator whose responses are evaluated")
    p.add_argument("--hf-model", default=None, help="override the Hugging Face model id")
    p.add_argument("--data-dir", default="data/ragtruth")
    p.add_argument("--split", default="test")
    p.add_argument("--tasks", default="QA,Summary,Data2txt")
    p.add_argument("--limit", type=int, default=0, help="max answers per task (0 = all)")
    p.add_argument("--labelled-only", action="store_true",
                   help="keep only responses with at least one hallucination span")
    p.add_argument("--n-samples", type=int, default=10)
    p.add_argument("--temperature", type=float, default=0.7)
    p.add_argument("--max-new-tokens", type=int, default=300)
    p.add_argument("--max-length", type=int, default=4096)
    p.add_argument("--max-distance", type=int, default=3, help="dependency-hop threshold for edges")
    p.add_argument("--prior-weight", type=float, default=2.0, help="subjective-logic prior weight W")
    p.add_argument("--n-boot", type=int, default=1000)
    p.add_argument("--load-in-4bit", action="store_true")
    p.add_argument("--reuse-dir", default=None, help="folder searched for earlier generations.jsonl and nli.jsonl files")
    p.add_argument("--time-budget-hours", type=float, default=0,
                   help="stop generating after this many hours so later stages still run (0 = no limit)")
    p.add_argument("--nli-model", default="microsoft/deberta-large-mnli")
    p.add_argument("--nli-batch", type=int, default=16)
    p.add_argument("--nli-strength", type=float, default=10.0,
                   help="how many resamples one NLI verdict is worth when fused (default: as many as --n-samples)")
    p.add_argument("--stage", choices=["all", "generate", "nli", "evaluate"], default="all")
    p.add_argument("--out", default="runs/pilot")
    args = p.parse_args()
    args.hf_model = args.hf_model or OPEN_MODELS[args.model_label]

    per_task = [load_ragtruth(args.data_dir, args.model_label, args.split, {task}, args.limit or None,
                              args.labelled_only)
                for task in args.tasks.split(",")]  # --limit applies per task, so tasks stay balanced
    # interleave tasks, so a session cut short by the time budget still covers every task evenly
    rows = [r for group in itertools.zip_longest(*per_task) for r in group if r is not None]
    print(f"{len(rows)} responses from {args.model_label} ({args.split})")
    os.makedirs(args.out, exist_ok=True)
    gen_path = os.path.join(args.out, "generations.jsonl")
    nli_path = os.path.join(args.out, "nli.jsonl")
    if args.stage in ("all", "generate"):
        stage_generate(args, rows, gen_path)
    if args.stage in ("all", "nli"):
        stage_nli(args, rows, gen_path, nli_path)
    if args.stage in ("all", "evaluate"):
        stage_evaluate(args, rows, gen_path, nli_path)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile full_study_analysis.py
"""Full-study analysis for Z-FMN (v3 outputs): element- and answer-level AUROC, paired tests, pooling.

Usage:
  python full_study_analysis.py Mistral-7B=runs/mistral Llama-2-7B=runs/llama7 Llama-2-13B=runs/llama13 > tables.md

Each argument is Model=folder, where the folder holds elements.csv (and responses.csv, for the
answer-level tables). Every model's rows are split by task, so a dataset is one (model, task) pair.
CIs come from a bootstrap that resamples answers (elements of one answer stay together); pooled
effects average the per-dataset differences with the bootstrap stratified by dataset. Measures
missing from a folder (for example NLI in pre-v3 runs) are left blank.
"""
import csv
import sys
from collections import defaultdict

import warnings

import numpy as np

warnings.filterwarnings("ignore", category=RuntimeWarning)

NB = 2000
rng = np.random.default_rng(7)

# score name -> (label, function of the column getter); higher score = more likely hallucinated
ELEMENT = {
    "logprob": ("Token log-prob", lambda c: -c("mean_lp")),
    "selfcheck": ("SelfCheckGPT-NLI", lambda c: c("selfcheck")),
    "nli_c": ("NLI contradiction", lambda c: c("nli_c")),
    "nli_ne": ("NLI not-entailed", lambda c: 1 - c("nli_e")),
    "P": ("Z-FMN P (model only)", lambda c: 1 - c("sl_P")),
    "u": ("Z-FMN u", lambda c: c("ctx_u")),
    "fP": ("Z-FMN + NLI P", lambda c: 1 - c("nli_fused_P")),
    "fd": ("Z-FMN + NLI d", lambda c: c("nli_fused_d")),
    "tP": ("Z-FMN + NLI + SelfCheck P", lambda c: 1 - c("tri_fused_P")),
    "freq": ("Resample frequency", lambda c: 1 - c("freq")),
}
ANSWER = {
    "logprob": ("Mean token log-prob", "resp_neg_mean_logprob"),
    "selfcheck": ("SelfCheckGPT-NLI", "resp_selfcheck_nli"),
    "nli_c": ("NLI max contradiction", "resp_nli_contra_max"),
    "nli_ne": ("NLI not-entailed", "resp_nli_not_entailed"),
    "P": ("Z-FMN max(1-P) edges", "resp_edge_P_max"),
    "Pn": ("Z-FMN max(1-P) nodes", "resp_node_P_max"),
    "fP": ("Z-FMN + NLI max(1-P) edges", "resp_edge_nli_fused_P_max"),
    "fPn": ("Z-FMN + NLI max(1-P) nodes", "resp_node_nli_fused_P_max"),
    "tP": ("Z-FMN + NLI + SelfCheck max(1-P) edges", "resp_edge_tri_fused_P_max"),
}
PAIRS = [("tP", "selfcheck"), ("tP", "fP"), ("P", "freq"), ("fP", "selfcheck"), ("fP", "nli_ne"), ("fP", "nli_c"), ("fP", "P"), ("P", "logprob"), ("selfcheck", "logprob")]


def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return np.nan


def read(path):
    try:
        with open(path) as f:
            return list(csv.DictReader(f))
    except FileNotFoundError:
        return []


def auc_boot(s, y, rid, W):
    """AUROC of s for labels y on the full sample, then on each bootstrap weight row of W."""
    m = ~np.isnan(s)
    s, y, rid = s[m], y[m], rid[m]
    if y.sum() == 0 or y.sum() == len(y):
        return np.nan, np.full(len(W), np.nan)
    o = np.argsort(s, kind="mergesort")
    s, y, rid = s[o], y[o], rid[o]
    _, g = np.unique(s, return_inverse=True)
    ng = g.max() + 1
    w = np.vstack([np.ones(len(s)), W[:, rid]])
    out = np.empty(len(w))
    for b in range(len(w)):
        pos = np.bincount(g[y == 1], w[b, y == 1], ng)
        neg = np.bincount(g[y == 0], w[b, y == 0], ng)
        cn = np.cumsum(neg) - neg
        den = pos.sum() * neg.sum()
        out[b] = (pos * (cn + .5 * neg)).sum() / den if den else np.nan
    return out[0], out[1:]


def datasets():
    for arg in sys.argv[1:]:
        model, folder = arg.split("=", 1)
        el, rs = read(f"{folder}/elements.csv"), read(f"{folder}/responses.csv")
        for task in sorted({r["task_type"] for r in el}):
            yield model, task, [r for r in el if r["task_type"] == task], [r for r in rs if r["task_type"] == task]


res = {}  # (dataset, level, kind, subset, measure) -> (point, boot)
counts = {}
names = []
for model, task, el, rs in datasets():
    name = f"{model} / {task}"
    names.append(name)
    ids = sorted({r["response_id"] for r in el} | {r["response_id"] for r in rs})
    pos = {k: i for i, k in enumerate(ids)}
    W = np.stack([np.bincount(rng.integers(0, len(ids), len(ids)), minlength=len(ids)) for _ in range(NB)])

    rid = np.array([pos[r["response_id"]] for r in el])
    y = np.array([int(r["wrong"]) for r in el])
    kind = np.array([r["kind"] for r in el])
    types = np.array([r["types"] or "" for r in el])
    col = lambda k: np.array([num(r.get(k)) for r in el])  # noqa: E731
    subsets = {"all": np.ones(len(el), bool),
               "conflict": (y == 0) | (np.char.find(types, "Conflict") >= 0),
               "baseless": (y == 0) | (np.char.find(types, "Baseless") >= 0)}
    for kd in ("edge", "node"):
        for sb, sm in subsets.items():
            m = (kind == kd) & sm
            counts[(name, kd, sb)] = (int(y[m].sum()), int(m.sum()))
            for key, (_, f) in ELEMENT.items():
                res[(name, "el", kd, sb, key)] = auc_boot(f(col)[m], y[m], rid[m], W)

    if rs:
        ra = np.array([pos[r["response_id"]] for r in rs])
        ya = np.array([int(r["wrong"]) for r in rs])
        counts[(name, "answer")] = (int(ya.sum()), len(ya))
        for key, (_, c) in ANSWER.items():
            res[(name, "ans", "-", "all", key)] = auc_boot(np.array([num(r.get(c)) for r in rs]), ya, ra, W)
    print(name, "elements", len(el), "answers", len(rs), file=sys.stderr)

ci = lambda b: (np.nanpercentile(b, 2.5), np.nanpercentile(b, 97.5)) if np.isfinite(b).any() else (np.nan, np.nan)  # noqa: E731
star = lambda lo, hi: " *" if lo > 0 or hi < 0 else ""  # noqa: E731


def cell(p, b):
    if np.isnan(p):
        return ""
    lo, hi = ci(b)
    return f"{p:.3f} [{lo:.2f}, {hi:.2f}]"


def dcell(p, b):
    if np.isnan(p) or not np.isfinite(b).any():
        return ""
    lo, hi = ci(b)
    return f"{p:+.3f} [{lo:+.3f}, {hi:+.3f}]{star(lo, hi)}"


def pooled(group, level, kd, sb, a, b):
    """Mean over datasets of AUROC(a) − AUROC(b), stratified bootstrap; datasets missing either are skipped."""
    ds = [n for n in group if (n, level, kd, sb, a) in res and not np.isnan(res[(n, level, kd, sb, a)][0])
          and not np.isnan(res[(n, level, kd, sb, b)][0])]
    if not ds:
        return np.nan, np.array([np.nan])
    p = np.mean([res[(n, level, kd, sb, a)][0] - res[(n, level, kd, sb, b)][0] for n in ds])
    bs = np.nanmean([res[(n, level, kd, sb, a)][1] - res[(n, level, kd, sb, b)][1] for n in ds], axis=0)
    return p, bs


L = []
L.append("## Table A. Element-level AUROC, all hallucinations (95% CI, bootstrap over answers)\n")
L.append("| Dataset | Element | " + " | ".join(v[0] for v in ELEMENT.values()) + " | Wrong / total |")
L.append("| --- | --- | " + " | ".join("---" for _ in ELEMENT) + " | --- |")
for n in names:
    for kd in ("edge", "node"):
        w, t = counts[(n, kd, "all")]
        L.append(f"| {n} | {kd} | " + " | ".join(cell(*res[(n, "el", kd, "all", k)]) for k in ELEMENT) + f" | {w} / {t} |")

tasks = sorted({n.split(" / ")[1] for n in names})
groups = [("All datasets", names)] + [(f"Task {t}", [n for n in names if n.endswith(" / " + t)]) for t in tasks]
L.append("\n## Table B. Pooled paired differences, element level (mean of per-dataset AUROC differences; * = CI excludes 0)\n")
L.append("| Pool | Element | Subset | " + " | ".join(f"{ELEMENT[a][0]} − {ELEMENT[b][0]}" for a, b in PAIRS) + " |")
L.append("| --- | --- | --- | " + " | ".join("---" for _ in PAIRS) + " |")
for gname, g in groups:
    for kd in ("edge", "node"):
        for sb in ("all", "conflict", "baseless"):
            L.append(f"| {gname} | {kd} | {sb} | " + " | ".join(dcell(*pooled(g, "el", kd, sb, a, b)) for a, b in PAIRS) + " |")

L.append("\n## Table C. Edge − node (same measure, pooled over all datasets)\n")
L.append("| Measure | Edge − node |")
L.append("| --- | --- |")
for k in ("P", "fP", "tP", "selfcheck", "nli_ne"):
    ds = [n for n in names if not np.isnan(res[(n, "el", "edge", "all", k)][0])]
    if ds:
        p = np.mean([res[(n, "el", "edge", "all", k)][0] - res[(n, "el", "node", "all", k)][0] for n in ds])
        bs = np.nanmean([res[(n, "el", "edge", "all", k)][1] - res[(n, "el", "node", "all", k)][1] for n in ds], axis=0)
        L.append(f"| {ELEMENT[k][0]} | {dcell(p, bs)} |")

ans = [n for n in names if (n, "answer") in counts and 0 < counts[(n, "answer")][0] < counts[(n, "answer")][1]]
if ans:
    L.append("\n## Table D. Answer-level AUROC: does the answer contain any hallucination? (needs clean answers)\n")
    L.append("| Dataset | " + " | ".join(v[0] for v in ANSWER.values()) + " | Hallucinated / answers |")
    L.append("| --- | " + " | ".join("---" for _ in ANSWER) + " | --- |")
    for n in ans:
        w, t = counts[(n, "answer")]
        L.append(f"| {n} | " + " | ".join(cell(*res[(n, "ans", "-", "all", k)]) for k in ANSWER) + f" | {w} / {t} |")
    apairs = [("tP", "selfcheck"), ("fP", "selfcheck"), ("fP", "nli_ne"), ("fP", "P"), ("P", "logprob")]
    L.append("\n## Table E. Pooled paired differences, answer level\n")
    L.append("| Pool | " + " | ".join(f"{ANSWER[a][0]} − {ANSWER[b][0]}" for a, b in apairs) + " |")
    L.append("| --- | " + " | ".join("---" for _ in apairs) + " |")
    for gname, g in groups:
        g = [n for n in g if n in ans]
        if g:
            L.append(f"| {gname} | " + " | ".join(dcell(*pooled(g, "ans", "-", "all", a, b)) for a, b in apairs) + " |")
print("\n".join(L))


In [ ]:
%%writefile rq4_emotion_analysis.py
"""RQ4: does epistemic reliability covary with the emotional content EmoAtlas measures?

Usage:
  python rq4_emotion_analysis.py Mistral-7B=runs/mistral Llama-2-7B=runs/llama7 ... > rq4_tables.md

Uses the node rows of each folder's elements.csv (EmoAtlas features exist for concepts only).

Model 1 (reliability). Linear mixed model, random intercept per answer:
  logit(P) ~ |valence| + emotion count + frame valence spread + Zipf word frequency + word length
             + log(1 + degree) + task + model
Model 2 (detection). Logistic regression with answer-clustered standard errors:
  hallucinated ~ (1 − P) × emotional + controls
  The interaction asks whether confidence detects hallucinations worse for emotionally loaded concepts.
Continuous predictors are z-scored, so coefficients are per standard deviation.
"""
import sys
import warnings

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from wordfreq import zipf_frequency

warnings.filterwarnings("ignore")

frames = []
for arg in sys.argv[1:]:
    model, folder = arg.split("=", 1)
    d = pd.read_csv(f"{folder}/elements.csv")
    d = d[d["kind"] == "node"].copy()
    d["model"] = model
    frames.append(d)
d = pd.concat(frames, ignore_index=True)
d["answer"] = d["model"] + ":" + d["response_id"].astype(str)
P = d["sl_P"].clip(1e-3, 1 - 1e-3)
d["logit_P"] = np.log(P / (1 - P))
d["risk"] = 1 - d["sl_P"]
d["abs_valence"] = d["valence"].abs()
d["emotional"] = ((d["abs_valence"] > 0) | (d["n_emotions"] > 0)).astype(int)
d["zipf"] = [zipf_frequency(str(k), "en") for k in d["key"]]
d["length"] = d["key"].astype(str).str.len()
d["log_degree"] = np.log1p(d["degree"])
d = d.dropna(subset=["abs_valence", "n_emotions", "frame_valence_spread", "logit_P"])
for c in ("abs_valence", "n_emotions", "frame_valence_spread", "zipf", "length", "log_degree", "risk"):
    d[c + "_z"] = (d[c] - d[c].mean()) / (d[c].std() or 1)
multi = lambda c: f" + C({c})" if d[c].nunique() > 1 else ""  # noqa: E731
controls = "zipf_z + length_z + log_degree_z" + multi("task_type") + multi("model")

L = [f"Nodes: {len(d)} from {d['answer'].nunique()} answers; emotional nodes: {int(d['emotional'].sum())}; "
     f"hallucinated nodes: {int(d['wrong'].sum())}.\n"]

m1 = smf.mixedlm(f"logit_P ~ abs_valence_z + n_emotions_z + frame_valence_spread_z + {controls}",
                 d, groups=d["answer"]).fit(reml=True)
L.append("## RQ4 Model 1. Reliability (logit P) and emotional content, linear mixed model\n")
L.append("| Term | Coefficient (per SD) | 95% CI | p |")
L.append("| --- | --- | --- | --- |")
ci = m1.conf_int()
for t in ("abs_valence_z", "n_emotions_z", "frame_valence_spread_z", "zipf_z", "length_z", "log_degree_z"):
    L.append(f"| {t[:-2]} | {m1.params[t]:+.3f} | [{ci.loc[t, 0]:+.3f}, {ci.loc[t, 1]:+.3f}] | {m1.pvalues[t]:.3g} |")

m2 = smf.logit(f"wrong ~ risk_z * emotional + {controls}", d).fit(
    disp=0, cov_type="cluster", cov_kwds={"groups": pd.factorize(d["answer"])[0]})
L.append("\n## RQ4 Model 2. Does emotional content change how well confidence detects hallucination?\n")
L.append("Logistic regression, standard errors clustered by answer. Odds ratios per SD of (1 − P).\n")
L.append("| Term | Odds ratio | 95% CI | p |")
L.append("| --- | --- | --- | --- |")
ci = m2.conf_int()
for t in ("risk_z", "emotional", "risk_z:emotional"):
    L.append(f"| {t} | {np.exp(m2.params[t]):.3f} | [{np.exp(ci.loc[t, 0]):.3f}, {np.exp(ci.loc[t, 1]):.3f}] | {m2.pvalues[t]:.3g} |")

# descriptive check: AUROC of 1 − P on emotional vs neutral nodes
def auroc(s, y):
    r = pd.Series(s).rank().to_numpy()
    n1 = y.sum()
    n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0) if n1 and n0 else np.nan


L.append("\n| Nodes | AUROC of 1 − P | Hallucinated / total |")
L.append("| --- | --- | --- |")
for lab, g in (("emotional", d[d["emotional"] == 1]), ("neutral", d[d["emotional"] == 0])):
    L.append(f"| {lab} | {auroc(g['risk'].to_numpy(), g['wrong'].to_numpy()):.3f} | {int(g['wrong'].sum())} / {len(g)} |")
print("\n".join(L))


## Log in to Hugging Face

In [ ]:
import os
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except Exception:
        pass
if token:
    os.environ['HF_TOKEN'] = token
    print('HF token loaded')
else:
    from huggingface_hub import notebook_login
    notebook_login()

## Quick check (should print five `ok` lines)

In [ ]:
!python tests/test_core.py

## Settings
`GEN_HOURS` is how long this session spends generating; the rest of the 12-hour Kaggle limit is left for
the entailment check and the evaluation. Models run in this order and each picks up where the last session stopped.

In [ ]:
MODELS = ['llama-2-7b-chat', 'mistral-7B-instruct', 'llama-2-13b-chat']
GEN_HOURS = 7
REUSE = '/kaggle/input' if os.path.exists('/kaggle/input') else 'runs'
COMMON = '--split test --tasks QA,Summary,Data2txt --n-samples 10 --load-in-4bit'
import json, time
def progress():
    for m in MODELS:
        g = f'runs/full_{m}/generations.jsonl'
        n = sum(1 for _ in open(g)) if os.path.exists(g) else 0
        print(f'{m}: {n} / 450 answers generated')

## 1. Reuse earlier work, then generate (GPU)

In [ ]:
t0 = time.time()
for m in MODELS:
    left = GEN_HOURS - (time.time() - t0) / 3600
    if left < 0.1:
        print('time budget used up; the next session continues with', m)
        break
    lh = round(left, 2)
    !python run_pilot.py --model-label {m} {COMMON} --stage generate --reuse-dir {REUSE} --time-budget-hours {lh} --out runs/full_{m}
progress()

## 2. Entailment check and SelfCheckGPT baseline (GPU, about 3 seconds per answer)

In [ ]:
for m in MODELS:
    if os.path.exists(f'runs/full_{m}/generations.jsonl'):
        !python run_pilot.py --model-label {m} {COMMON} --stage nli --reuse-dir {REUSE} --out runs/full_{m}

## 3. Evaluate each model (CPU)

In [ ]:
for m in MODELS:
    if os.path.exists(f'runs/full_{m}/generations.jsonl'):
        !python run_pilot.py --model-label {m} {COMMON} --stage evaluate --n-boot 1000 --out runs/full_{m}

## 4. Cross-model tables (paired tests and pooled effects)

In [ ]:
args = ' '.join(f'{m}=runs/full_{m}' for m in MODELS if os.path.exists(f'runs/full_{m}/elements.csv'))
!python full_study_analysis.py {args} > runs/full_study_tables.md
progress()
print(open('runs/full_study_tables.md').read())

## 5. RQ4: emotional content and reliability (mixed models)

In [ ]:
!python rq4_emotion_analysis.py {args} > runs/rq4_tables.md
print(open('runs/rq4_tables.md').read())

## Download
After the version finishes, download `zfmn-results.zip` from the Output tab and send it in the project thread.

In [ ]:
import shutil
shutil.make_archive('zfmn-results', 'zip', 'runs')
print('saved zfmn-results.zip')